# Exercise 8 · A failing check blocks the merge

You make the `checks` check required on `main`. Then you push a broken change and see GitHub refuse to merge it.

**Goal:** A ruleset on `main` requires a pull request and the `checks` check. A pull request with a failing check cannot merge until you fix it.

**Why it matters:** A red check only warns. A required check stops the merge.

**Before you start:** You did exercise 7 in the past seven days: your public `ci-practice` repository runs `checks` on each pull request. GitHub requires a check that passed in that time.

**What the check measures:** on GitHub, `main` requires a pull request and `checks`. Your newest pull request failed `checks`, then passed, and you merged it. Your answer to Step 7's question is right.

**Done when:** the last cell prints `Done:`.

## The situation

A teammate's change breaks `filter_cases`: it returns every case. `broken/cases.py` is that change. The file holds made-up code.

## Step 1 · Write the practice file

Run the cell. It writes `broken/cases.py` into `practice/`, beside this notebook. It never overwrites a file.

Expect `Wrote practice/broken/cases.py`, or `Kept` if it is already there.

In [ ]:
FILES = {
    "broken/cases.py": """\"\"\"Pick the cases that have one status.\"\"\"


def filter_cases(cases: list[dict], status: str) -> list[dict]:
    \"\"\"Return the cases that have one status, in their order.

    Args:
        cases: The cases. Each one is a dict with a "status" key.
        status: The status to keep.

    Returns:
        The cases whose status is status, in their order.
    \"\"\"
    return list(cases)
""",
}

from pathlib import Path

PRACTICE = Path("practice")
for name, text in FILES.items():
    path = PRACTICE / name
    if path.exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

## Step 2 · Clone your repository

Choose **Terminal > New Terminal**. Type each line, then press Enter. Use your account name:

```sh
source .venv/bin/activate
cd week-02/day-03/exercise-08-a-failing-check-blocks-the-merge/practice
git clone https://github.com/<your-account>/ci-practice.git
cd ci-practice
git log --oneline -1
```

Expect `Merge pull request #1` from exercise 7.

## Step 3 · Require the check on main

On your repository's page on github.com:

1. Click **Settings**. In the left sidebar, click **Rules**, then **Rulesets**.
2. Click **New ruleset**, then **New branch ruleset**.
3. Under "Ruleset name", type `protect-main`. Change the enforcement status from **Disabled** to **Active**.
4. Under "Target branches", click **Add a target**, then **Include default branch**.
5. Under "Branch protections", tick **Require a pull request before merging** and **Require status checks to pass before merging**.
6. In that rule's additional settings, enter `checks` as the status check, then click **+** to add it.
7. Click **Create**.

Expect `protect-main` in the list of rulesets.

### Learn: only a required check blocks a merge

Without the rule, "collaborators can merge the branch at any time". With it, "all required status checks must pass before collaborators can merge changes into the branch" ([Available rules for rulesets](https://docs.github.com/en/repositories/configuring-branches-and-merges-in-your-repository/managing-rulesets/available-rules-for-rulesets)).

## Step 4 · Push the broken change and open a pull request

```sh
git switch -c try-broken-filter
cp ../broken/cases.py cases.py
python -m pytest -q
git commit -qam "Try the broken filter"
git push -u origin try-broken-filter
```

Expect `2 failed`. You push it anyway, on purpose. On github.com, click **Compare & pull request**, then **Create pull request**.

Expect the `checks` check to fail after about a minute. Now try to merge: GitHub refuses while `checks` fails.

## Step 5 · Read the failing test in the log

Click **Actions**, then the newest run, then the job `checks`. Failed steps open by themselves.

Expect `FAILED tests/test_cases.py::test_open_returns_only_open_cases` in the step that runs `pytest`.

## Step 6 · Undo the broken change, then merge

`git restore --source=main` takes `cases.py` as it is on `main`:

```sh
git restore --source=main cases.py
git commit -qam "Restore the filter"
git push
```

On the pull request, wait for `checks` to pass. Click **Merge pull request**, then `Confirm merge`. Then:

```sh
git switch main
git pull
git log --oneline -1
```

Expect `Merge pull request #2`. Your number can differ.

## Step 7 · Answer one question

A pull request's check fails, and `main` has no rule. What does the failing check stop?

- **A.** The merge.
- **B.** Nothing. Anyone can still merge.
- **C.** The push to the branch.

Write your letter between the quotes, then run the cell.

<details><summary>Show the answer</summary>

B. Why: a check only reports. The ruleset is what refused your merge in Step 4.

</details>

In [ ]:
ANSWER = ""
print(f"Saved your answer: {ANSWER}")

## Check your work

Write your repository in `REPO`, then run the cell. It reads GitHub three times at most, and saves the replies in `practice/github.json` for reruns. Expect four `ok:` lines, then `Done:`.

In [ ]:
# Check your work: main's rules, your newest pull request and its runs on GitHub (3 public reads at most)
REPO = ""  # your repository, for example "your-account/ci-practice"
REFRESH = False  # True: ask GitHub again instead of reusing practice/github.json

import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

SAVED = Path("practice/github.json")
AGAIN = " Then set REFRESH = True and rerun."


def read(path: str, saved: dict) -> object:
    """Return GitHub's reply for path: the saved one, or a public read, then saved."""
    if path not in saved["reads"]:
        ask = urllib.request.Request("https://api.github.com" + path, headers={"User-Agent": "course-check"})
        try:
            with urllib.request.urlopen(ask, timeout=30) as reply:
                saved["reads"][path] = json.load(reply)
        except urllib.error.HTTPError as error:
            if error.code in (403, 429):
                reset = time.localtime(int(error.headers.get("x-ratelimit-reset") or time.time() + 3600))
                raise RuntimeError(f"GitHub allows 60 reads an hour per address, shared by your classroom. "
                                   f"Try again after {time.strftime('%H:%M', reset)}.")
            raise RuntimeError(f"GitHub answered {error.code} for {saved['repo']}. Check REPO, and that the "
                               "repository is public (exercise 7).")
        except (urllib.error.URLError, OSError):
            raise RuntimeError("the check could not reach GitHub. Check your internet connection, then run it again.")
        SAVED.write_text(json.dumps(saved, indent=1), encoding="utf-8")
    return saved["reads"][path]


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    found = re.search(r"([\w.-]+)/([\w.-]+?)(?:\.git)?/?$", REPO.strip())
    if not found:
        return "REPO is empty. Write your repository at the top of this cell, like your-account/ci-practice."
    repo = f"{found[1]}/{found[2]}"
    saved = json.loads(SAVED.read_text(encoding="utf-8")) if SAVED.is_file() and not REFRESH else {}
    if saved.get("repo") != repo:
        saved = {"repo": repo, "reads": {}}
    rules = read(f"/repos/{repo}/rules/branches/main", saved)
    required = [check.get("context") for rule in rules if rule.get("type") == "required_status_checks"
                for check in rule.get("parameters", {}).get("required_status_checks", [])]
    if "pull_request" not in [rule.get("type") for rule in rules] or "checks" not in required:
        return "main has no active rule that requires a pull request and the checks check (Step 3)." + AGAIN
    print("ok: main requires a pull request and the checks check")
    pulls = read(f"/repos/{repo}/pulls?state=all&per_page=1", saved)
    if not pulls:
        return "your repository has no pull request. Open one with the broken filter (Step 4)." + AGAIN
    number, branch, head = pulls[0]["number"], pulls[0]["head"]["ref"], pulls[0]["head"]["sha"]
    query = urllib.parse.urlencode({"branch": branch, "event": "pull_request", "per_page": 20})
    runs = [run for run in read(f"/repos/{repo}/actions/runs?{query}", saved)["workflow_runs"]
            if run.get("path", "").endswith("checks.yml")]
    if not any(run.get("conclusion") == "failure" for run in runs):
        return f"pull request #{number} never failed its checks. Push the broken filter on a new branch (Step 4)." + AGAIN
    print(f"ok: the checks run failed on pull request #{number}")
    if not any(run.get("head_sha") == head and run.get("conclusion") == "success" for run in runs):
        return f"the last commit of pull request #{number} has not passed its checks yet (Step 6)." + AGAIN
    if not pulls[0].get("merged_at"):
        return f"pull request #{number} is not merged. Merge it (Step 6)." + AGAIN
    print(f"ok: its last commit passed, and pull request #{number} is merged")
    if str(globals().get("ANSWER", "")).strip().upper() != "B":
        return "the question in Step 7 has no right answer yet. Read the Learn cell after Step 3."
    print("ok: your answer is right")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = str(error) if isinstance(error, RuntimeError) else f"the check stopped on {error!r}. Run Step 1, then this cell."
print(f"Not yet: {problem}" if problem else
      "Done: main refused a change while its check failed, and took it once the check passed.")

## What you learned

- A red check warns. Only a rule that requires it stops the merge.
- The proof that a rule works is a change it refused.
- A deployment follows the same rule: a change reaches the next environment, like `staging`, only after its checks pass.

**Reflect:** Which branch in your own work would you protect first, and with which check?